# Fire Season Timing | Mediterranean Basin

In [ ]:
'''
Computes fire season timing metrics (onset, peak, end, season length) for all WWF RESOLVE ecoregions
intersecting the study region for years 2003-2025.

Data sources:
- MODIS Terra active fire: MODIS/061/MOD14A1
- MODIS Aqua active fire:  MODIS/061/MYD14A1
- Ecoregions:              RESOLVE/ECOREGIONS/2017

Region definition:
- Mediterranean Basin bounding box: lon -10 to 42, lat 28 to 48
- Ecoregion number can be changed by parameters
- Subsetting: use TEST_N / TEST_IDS in Cell 10 to run on a reduced set


Output (all paths derived from RUN_LABEL, RUN_VERSION, and BASE_OUT_DIR):
- <BASE_OUT_DIR>/outputs/<RUN_LABEL>_<RUN_VERSION>/<ECO_ID>_<ECO_NAME>.csv
- <BASE_OUT_DIR>/outputs/<RUN_LABEL>_<RUN_VERSION>/daily_counts/
- <BASE_OUT_DIR>/outputs/<RUN_LABEL>_<RUN_VERSION>/_all_metrics.csv
- <BASE_OUT_DIR>/outputs/<RUN_LABEL>_<RUN_VERSION>/_all_daily_counts.csv
- <BASE_OUT_DIR>/outputs/<RUN_LABEL>_<RUN_VERSION>/master_<RUN_LABEL>_<RUN_VERSION>.csv
- <BASE_OUT_DIR>/outputs/<RUN_LABEL>_<RUN_VERSION>/README.txt
- <BASE_OUT_DIR>/outputs/<RUN_LABEL>_<RUN_VERSION>/eco_geometries.json
'''

import ee
import pandas as pd
import numpy as np
import os
import time
import datetime
import calendar
import json
from tqdm import tqdm

In [ ]:
# Authenticate and initialize ----------------------------------------------------------------------
ee.Authenticate()
ee.Initialize(project='fire-seasons')

In [ ]:
# RUN CONFIGURATION --------------------------------------------------------------------------------
# Set these before running anything else. All output paths are derived from these values.

RUN_LABEL   = 'med_basin'  # short name for this run
RUN_VERSION = 'v4'         # increment this for each new run
RUN_NOTES   = """
Testing new folder stucture
"""

In [ ]:
# Folder structure and paths -----------------------------------------------------------------------

BASE_OUT_DIR = r'C:\Users\ibekar\Documents\GitProjects\TGPF'  # Windows
# BASE_OUT_DIR = '/Users/ibekar/Github/TGPF'                  # Mac

_run_stamp = datetime.date.today().strftime('%Y-%m-%d')
_run_name  = f'{RUN_LABEL}_{RUN_VERSION}'
run_dir    = os.path.join(BASE_OUT_DIR, 'runs', _run_name)
raw_dir    = os.path.join(run_dir, 'raw')
output_dir = os.path.join(run_dir, 'fire_metrics')
daily_dir  = os.path.join(output_dir, 'daily_counts')

os.makedirs(output_dir, exist_ok=True)

print(f'Run name  : {_run_name}')
print(f'Run dir   : {run_dir}')
print(f'Raw dir   : {raw_dir}')
print(f'Output dir: {output_dir}')

## Setup

In [ ]:
# LOAD MODIS COLLECTIONS ---------------------------------------------------------------------------
# Terra and Aqua are loaded once here at module level.
# Per-year and per-day filtering is handled inside get_daily_counts().

terra = ee.ImageCollection("MODIS/061/MOD14A1").select('FireMask')
aqua  = ee.ImageCollection("MODIS/061/MYD14A1").select('FireMask')

print('Terra image count:', terra.size().getInfo())
print('Aqua image count:', aqua.size().getInfo())
print('Terra and Aqua collections loaded.')

In [ ]:
# LOAD MEDITERRANEAN BASIN ECOREGIONS --------------------------------------------------------------
med_bbox = ee.Geometry.BBox(-10, 28, 42, 48)

ecoregions_med = ee.FeatureCollection("RESOLVE/ECOREGIONS/2017").filterBounds(med_bbox)

n_eco    = ecoregions_med.size().getInfo()
eco_list = ecoregions_med.select(['ECO_ID', 'ECO_NAME', 'BIOME_NUM', 'BIOME_NAME']).getInfo()

print(f'Number of ecoregions intersecting Mediterranean bounding box: {n_eco}')
print()
for f in eco_list['features']:
    p = f['properties']
    print(p['ECO_ID'], '|', p['ECO_NAME'], '|', p['BIOME_NAME'])

In [ ]:
# BUILD ECO RECORDS --------------------------------------------------------------------------------
eco_records = []
for f in eco_list['features']:
    p = f['properties']
    eco_records.append({
        'eco_id'    : p['ECO_ID'],
        'eco_name'  : p['ECO_NAME'],
        'biome_num' : p['BIOME_NUM'],
        'biome_name': p['BIOME_NAME'],
        'geometry'  : ee.Geometry(f['geometry'])
    })

print(f'Built {len(eco_records)} ecoregion records.')

## Parameters

In [ ]:
# PARAMETERS ---------------------------------------------------------------------------------------

FIRE_MASK_MIN   = 8     # FireMask threshold: >= 8 = nominal + high confidence only
ONSET_THRESHOLD = 0.05  # Cumulative fraction threshold for fire season onset (5%)
END_THRESHOLD   = 0.95  # Cumulative fraction threshold for fire season end (95%)
MIN_DETECTIONS  = 20    # Minimum annual fire detections required to compute metrics
YEARS           = list(range(2003, 2026))  # Full study period: 2003–2025

# BIMODALITY DIAGNOSTICS ---------------------------------------------------------------------------
# Applied only when season_length > MIN_SEASON_FOR_BIMODALITY days.
# Soft flag: one metric triggers. Hard flag: both trigger.
MIN_SEASON_FOR_BIMODALITY = 90    # Minimum season length (days) before bimodality is assessed
BC_THRESHOLD              = 0.555 # Bimodality coefficient above this → bimodality signal

In [ ]:
# SUBSETTING (set to None to disable) --------------------------------------------------------------
TEST_N   = 2
TEST_IDS = None

# APPLY SUBSETTING ---------------------------------------------------------------------------------
eco_run = eco_records

if TEST_IDS is not None:
    eco_run = [e for e in eco_run if e['eco_id'] in TEST_IDS]
    print(f'Subsetting to {len(eco_run)} ecoregions by ID: {TEST_IDS}')

if TEST_N is not None:
    eco_run = eco_run[:TEST_N]
    print(f'Subsetting to first {TEST_N} ecoregions.')

print(f'Running pipeline on {len(eco_run)} / {len(eco_records)} ecoregions.')

In [ ]:
# SAVE GEOMETRIES TO DISK --------------------------------------------------------------------------
# Saves ecoregion geometries as GeoJSON for reuse in visualization notebooks
# without needing a GEE connection. Skipped if file already exists.
# Uses eco_run and respects subsetting if active, full list if not.

os.makedirs(run_dir, exist_ok=True)
geo_path = os.path.join(run_dir, 'eco_geometries.json')

if os.path.exists(geo_path):
    print(f'Geometries already saved. Skipping. ({geo_path})')
else:
    geo_records_export = []
    for rec in eco_run:
        geo_records_export.append({
            'eco_id'  : rec['eco_id'],
            'eco_name': rec['eco_name'],
            'geometry': rec['geometry'].getInfo()
        })

    with open(geo_path, 'w') as f:
        json.dump(geo_records_export, f)

    print(f'Saved {len(geo_records_export)} geometries → {geo_path}')

In [ ]:
# WRITE README -------------------------------------------------------------------------------------
_readme_path = os.path.join(run_dir, 'README.txt')
with open(_readme_path, 'w') as _f:
    _f.write(f'Run name    : {_run_name}\n')
    _f.write(f'Date        : {_run_stamp}\n')
    _f.write(f'Years       : {YEARS[0]}–{YEARS[-1]}\n')
    _f.write(f'TEST_N      : {TEST_N}\n')
    _f.write(f'TEST_IDS    : {TEST_IDS}\n')
    _f.write(f'Ecoregions  : {len(eco_run)} / {len(eco_records)}\n')
    _f.write(f'\nNotes:\n{RUN_NOTES.strip()}\n')
print(f'README written → {_readme_path}')

## Helper Functions

In [ ]:
# FUNCTION: get_daily_counts -----------------------------------------------------------------------


def get_daily_counts(eco_geometry, year):
    """
    Compute daily MODIS active fire detection counts for a given
    ecoregion geometry and calendar year.

    Combines Terra (MOD14A1) and Aqua (MYD14A1) by taking the pixel-wise
    maximum across sensors for each day, deduplicating detections that
    appear in both sensors on the same day.

    All 365 daily counts are retrieved in a SINGLE reduceRegion call
    by stacking all daily images into one multi-band image using toBands().
    This avoids the 'Too many concurrent aggregations' error that occurs
    when reduceRegion is called inside a mapped function.

    Parameters
    ----------
    eco_geometry : ee.Geometry
        The geometry of the ecoregion to compute counts for.
    year : int
        The calendar year to process (e.g. 2008).

    Returns
    -------
    pd.DataFrame
        DataFrame with columns:
          - doy           : int, day of year (1-indexed)
          - n_detections  : int, number of fire pixels detected
        One row per day of the year (365 or 366 rows), sorted by doy.
    """

    start  = ee.Date.fromYMD(year, 1, 1)
    end    = ee.Date.fromYMD(year + 1, 1, 1)
    n_days = 366 if calendar.isleap(year) else 365

    # Pre-filter both collections to this year
    terra_year = terra.filterDate(start, end)
    aqua_year  = aqua.filterDate(start, end)

    # Fallback empty image for days where a sensor returns no image
    empty = ee.Image.constant(0).rename('FireMask').toUint8()

    # Server-side list of day offsets: [0, 1, 2, ... n_days-1]
    day_seq = ee.List.sequence(0, n_days - 1)

    def make_daily_image(d):
        """
        For a single day offset d, build a deduplicated binary fire image.
        Returns a single-band image named by its DOY (e.g. 'day_001').
        No reduceRegion here — reduction happens once outside this function.
        """
        d        = ee.Number(d)
        date     = start.advance(d, 'day')
        date_end = date.advance(1, 'day')

        terra_day = terra_year.filterDate(date, date_end)
        aqua_day  = aqua_year.filterDate(date, date_end)

        # Use empty fallback if sensor has no image for this day
        t = ee.Image(ee.Algorithms.If(
            terra_day.size().gt(0),
            terra_day.select('FireMask').max(),
            empty
        ))
        a = ee.Image(ee.Algorithms.If(
            aqua_day.size().gt(0),
            aqua_day.select('FireMask').max(),
            empty
        ))

        # Pixel-wise max across sensors = deduplication
        combined    = t.max(a)
        fire_binary = combined.gte(FIRE_MASK_MIN).unmask(0)

        # Name this band by its DOY so we can identify it after toBands()
        band_name = ee.String('day_').cat(
            d.add(1).toInt().format('%03d')
        )

        return fire_binary.rename(band_name)

    # Build a collection of 365 single-band images
    daily_collection = ee.ImageCollection(day_seq.map(make_daily_image))

    # Stack all 365 bands into ONE multi-band image
    stacked = daily_collection.toBands()

    # ONE single reduceRegion call on the entire stacked image
    counts_dict = stacked.reduceRegion(
        reducer   = ee.Reducer.sum(),
        geometry  = eco_geometry,
        scale     = 1000,
        maxPixels = 1e9
    ).getInfo()

    # Parse back into a tidy DataFrame.
    # NOTE: do NOT use sorted() on the raw band names. toBands() prepends a
    # collection index to each band name, producing keys like '0_day_001',
    # '1_day_002', ..., '9_day_010', '10_day_011'. Lexicographic sort puts
    # '10_day_011' before '1_day_002', scrambling all but the first entry.
    # Instead: extract the DOY from the last segment of each key, then sort
    # numerically by that value.
    rows = []
    for band_name, count in counts_dict.items():
        doy = int(band_name.split('_')[-1])
        rows.append({
            'doy'         : doy,
            'n_detections': int(count) if count is not None else 0
        })

    return pd.DataFrame(rows).sort_values('doy').reset_index(drop=True)

In [ ]:
# FUNCTION: compute_timing_metrics -----------------------------------------------------------------

def compute_timing_metrics(df, year):
    """
    Derives all per-ecoregion-year timing metrics and quality indicators from a daily detection.

    Four sections:
      1. Primary metrics        — onset/peak/end/season_length at 5%/95% thresholds
      2. Alternative thresholds — onset/end recomputed at 10%/90% and 15%/85%
      3. Profile shape          — median DOY, mean-median divergence, IQR season
                                   length, active days, peak concentration, skewness
      4. Bimodality diagnostics — bimodality coefficient (BC), per-year bimodal flag
                                   (0=clean, 1=flagged)
                                   (only computed when season_length > MIN_SEASON_FOR_BIMODALITY)

    Returns None if total detections < MIN_DETECTIONS.
    """

    total = df['n_detections'].sum()
    if total < MIN_DETECTIONS:
        print(f'  {year}: insufficient detections ({total}), skipping.')
        return None

    df     = df.copy().sort_values('doy').reset_index(drop=True)
    doys   = df['doy'].values.astype(float)
    counts = df['n_detections'].values.astype(float)

    cumulative = df['n_detections'].cumsum()
    cum_frac   = cumulative / total

    # Helper: return the first DOY where cumulative fraction >= frac
    def doy_at_frac(frac):
        rows = df[cum_frac >= frac]
        return int(rows.iloc[0]['doy']) if not rows.empty else None

    # -----------------------------------------------------------------------
    # 1. PRIMARY TIMING METRICS (5% / 95%)
    # -----------------------------------------------------------------------
    onset_doy = doy_at_frac(ONSET_THRESHOLD)
    end_doy   = doy_at_frac(END_THRESHOLD)

    if onset_doy is None or end_doy is None:
        print(f'  {year}: could not compute onset or end, skipping.')
        return None

    # Peak: detection-weighted mean DOY (fire activity centroid)
    peak_doy = int(round((doys * counts).sum() / counts.sum()))

    season_length = end_doy - onset_doy + 1

    peak_outside_window = not (onset_doy <= peak_doy <= end_doy)
    if peak_outside_window:
        print(f'  {year}: WARNING — peak ({peak_doy}) outside onset-end window '
              f'({onset_doy}–{end_doy}), flagging.')

    onset_month = (datetime.date(year, 1, 1) + datetime.timedelta(days=onset_doy - 1)).month
    peak_month  = (datetime.date(year, 1, 1) + datetime.timedelta(days=peak_doy  - 1)).month

    # -----------------------------------------------------------------------
    # 2. ALTERNATIVE THRESHOLDS (10%/90% and 15%/85%)
    # -----------------------------------------------------------------------
    onset_doy_10 = doy_at_frac(0.10)
    end_doy_90   = doy_at_frac(0.90)
    onset_doy_15 = doy_at_frac(0.15)
    end_doy_85   = doy_at_frac(0.85)

    # -----------------------------------------------------------------------
    # 3. PROFILE SHAPE METRICS
    # -----------------------------------------------------------------------

    # Median DOY: DOY at which 50% of detections have occurred
    median_doy = doy_at_frac(0.50)

    # Mean-median divergence: how far the weighted mean is from the median (days)
    # Large divergence = asymmetric season = threshold metrics may be less reliable
    mean_median_div = abs(peak_doy - median_doy) if median_doy is not None else None

    # IQR season length: distance between 25th and 75th percentile DOY
    # More robust than primary season_length because it ignores sparse tails
    q25_doy = doy_at_frac(0.25)
    q75_doy = doy_at_frac(0.75)
    iqr_season_length = (q75_doy - q25_doy + 1) if (q25_doy is not None and
                                                      q75_doy is not None) else None

    # Active days: days with at least one detection within the onset–end window
    season_mask = (df['doy'] >= onset_doy) & (df['doy'] <= end_doy)
    active_days = int((df.loc[season_mask, 'n_detections'] > 0).sum())

    # Peak concentration: fraction of annual detections within ±45 days of peak
    # High = tight unimodal season; low = diffuse or potentially bimodal
    conc_mask          = (df['doy'] >= peak_doy - 45) & (df['doy'] <= peak_doy + 45)
    peak_concentration = round(float(df.loc[conc_mask, 'n_detections'].sum() / total), 4)

    # Weighted skewness: treats detection counts as weights, days as values
    # Positive = long right tail (slow end); negative = long left tail (slow start)
    # Also computes raw (Pearson) kurtosis needed for BC below
    w_mean = (doys * counts).sum() / counts.sum()
    diffs  = doys - w_mean
    w_var  = (counts * diffs**2).sum() / counts.sum()
    w_std  = np.sqrt(w_var)

    if w_std > 0:
        w_skewness     = round(float((counts * (diffs / w_std)**3).sum() / counts.sum()), 4)
        w_kurtosis_raw = round(float((counts * (diffs / w_std)**4).sum() / counts.sum()), 4)
    else:
        w_skewness     = None
        w_kurtosis_raw = None

    # -----------------------------------------------------------------------
    # 4. BIMODALITY DIAGNOSTICS
    # Only computed when season_length exceeds MIN_SEASON_FOR_BIMODALITY.
    # Short seasons don't have enough temporal spread for BC to behave stably.
    # -----------------------------------------------------------------------
    if season_length > MIN_SEASON_FOR_BIMODALITY:

        # Bimodality coefficient (BC)
        # BC = (skewness² + 1) / raw_kurtosis
        # Threshold 0.555 = BC of a uniform distribution.
        # BC > 0.555 suggests the distribution is more bimodal than uniform.
        if w_kurtosis_raw is not None and w_kurtosis_raw > 0:
            bc = round(float((w_skewness**2 + 1) / w_kurtosis_raw), 4)
        else:
            bc = None

        # Per-year bimodal flag: 1 = BC above threshold, 0 = clean
        bimodal_flag_year = 1 if (bc is not None and bc > BC_THRESHOLD) else 0

    else:
        # Season too short to assess bimodality reliably
        bc                = None
        bimodal_flag_year = 0

    # -----------------------------------------------------------------------
    # RETURN
    # -----------------------------------------------------------------------
    return {
        'year'                : year,
        # --- Primary metrics ---
        'onset_doy'           : onset_doy,
        'peak_doy'            : peak_doy,
        'end_doy'             : end_doy,
        'season_length'       : season_length,
        'n_detections'        : int(total),
        'onset_month'         : onset_month,
        'peak_month'          : peak_month,
        'peak_outside_window' : int(peak_outside_window),
        # --- Alternative thresholds ---
        'onset_doy_10'        : onset_doy_10,
        'end_doy_90'          : end_doy_90,
        'onset_doy_15'        : onset_doy_15,
        'end_doy_85'          : end_doy_85,
        # --- Profile shape ---
        'median_doy'          : median_doy,
        'mean_median_div'     : mean_median_div,
        'q25_doy'             : q25_doy,
        'q75_doy'             : q75_doy,
        'iqr_season_length'   : iqr_season_length,
        'active_days'         : active_days,
        'peak_concentration'  : peak_concentration,
        'skewness'            : w_skewness,
        'kurtosis_raw'        : w_kurtosis_raw,
        # --- Bimodality diagnostics ---
        'bc'                  : bc,
        'bimodal_flag_year'   : bimodal_flag_year,
    }

## Main Pipeline

In [ ]:
# FULL PIPELINE LOOP - ALL ECOREGIONS x ALL YEARS --------------------------------------------------

os.makedirs(output_dir, exist_ok=True)
os.makedirs(daily_dir,  exist_ok=True)

all_metrics  = []
failed_years = []

for eco in tqdm(eco_run, desc='Ecoregions'):
    eco_id    = eco['eco_id']
    eco_name  = eco['eco_name']
    biome_num = eco['biome_num']
    biome_name= eco['biome_name']
    geometry  = eco['geometry']

    safe_name  = eco_name.replace(' ', '_').replace('/', '_')
    eco_path   = os.path.join(output_dir, f'{eco_id}_{safe_name}.csv')
    daily_path = os.path.join(daily_dir,  f'{eco_id}_{safe_name}_daily.csv')

    # ------------------------------------------------------------------
    # CHECKPOINT — daily file is the single signal that this ecoregion
    # was fully processed. eco_path may be absent if no valid years exist.
    # ------------------------------------------------------------------
    if os.path.exists(daily_path):
        if os.path.exists(eco_path):
            existing = pd.read_csv(eco_path)
            all_metrics.extend(existing.to_dict('records'))
            print(f'  Skipping {eco_name} — already done')
        else:
            print(f'  {eco_name} — daily counts on disk but no metrics file, recomputing locally.')
            saved_daily = pd.read_csv(daily_path)
            eco_metrics = []

            for year in YEARS:
                df_year = saved_daily[saved_daily['year'] == year][['doy', 'n_detections']]
                if df_year.empty:
                    continue
                metrics = compute_timing_metrics(df_year, year)
                if metrics is not None:
                    metrics['eco_id']    = eco_id
                    metrics['eco_name']  = eco_name
                    metrics['biome_num'] = biome_num
                    metrics['biome_name']= biome_name
                    eco_metrics.append(metrics)
                    all_metrics.append(metrics)
                else:
                    failed_years.append({
                        'eco_id': eco_id, 'eco_name': eco_name,
                        'year': year, 'reason': 'metrics_none'
                    })

            n_years_valid   = len(eco_metrics)
            pct_years_valid = round(n_years_valid / len(YEARS), 3)
            for m in eco_metrics:
                m['n_years_valid']   = n_years_valid
                m['pct_years_valid'] = pct_years_valid

            if eco_metrics:
                pd.DataFrame(eco_metrics).to_csv(eco_path, index=False)
                print(f'  Recovered {len(eco_metrics)} metric years from daily file')
            else:
                print(f'  No valid fire years for {eco_name} — confirmed from daily file')

        continue

    # ------------------------------------------------------------------
    # GEE FETCH: only reaches here if daily_path does not exist
    # ------------------------------------------------------------------
    print(f'\n=== {eco_name} (ID: {eco_id}) ===')
    eco_metrics   = []
    daily_records = []

    for year in YEARS:
        t0 = time.time()

        try:
            df_year = get_daily_counts(geometry, year)
        except Exception as e:
            failed_years.append({
                'eco_id': eco_id, 'eco_name': eco_name,
                'year': year, 'reason': f'exception: {e}'
            })
            print(f'  {year}: ERROR — {e}')
            continue

        df_year['year']     = year
        df_year['eco_id']   = eco_id
        df_year['eco_name'] = eco_name
        daily_records.extend(df_year.to_dict('records'))

        metrics = compute_timing_metrics(df_year, year)

        if metrics is not None:
            metrics['eco_id']    = eco_id
            metrics['eco_name']  = eco_name
            metrics['biome_num'] = biome_num
            metrics['biome_name']= biome_name
            eco_metrics.append(metrics)
            all_metrics.append(metrics)
        else:
            failed_years.append({
                'eco_id': eco_id, 'eco_name': eco_name,
                'year': year, 'reason': 'metrics_none'
            })

        t1 = time.time()
        print(f'  {year}: done in {t1 - t0:.1f}s')

    # Save daily counts (always, even if all metric years failed)
    if daily_records:
        daily_df = pd.DataFrame(daily_records)[[
            'eco_id', 'eco_name', 'year', 'doy', 'n_detections'
        ]]
        daily_df.to_csv(daily_path, index=False)
        print(f'  Saved {len(daily_df)} daily rows → {os.path.abspath(daily_path)}')

    # Quality flags across all valid years for this ecoregion
    n_years_valid   = len(eco_metrics)
    pct_years_valid = round(n_years_valid / len(YEARS), 3)
    for m in eco_metrics:
        m['n_years_valid']   = n_years_valid
        m['pct_years_valid'] = pct_years_valid

    # Save metrics CSV
    if eco_metrics:
        eco_df = pd.DataFrame(eco_metrics)
        eco_df.to_csv(eco_path, index=False)
        print(f'  Saved {len(eco_metrics)} metric years → {os.path.abspath(eco_path)}')
    else:
        print(f'  No valid metric years for {eco_name}.')

    # Update failed log after each ecoregion
    if failed_years:
        pd.DataFrame(failed_years).to_csv(
            os.path.join(output_dir, '_failed.csv'), index=False
        )

print('\nAll ecoregions complete.')

## Post-Run Assembly

In [ ]:
# POST-RUN ASSEMBLY — METRICS AND DAILY COUNTS -----------------------------------------------------

import glob

# Assemble metrics
metric_files = sorted(glob.glob(os.path.join(output_dir, '[!_]*.csv')))
if metric_files:
    metrics_combined = pd.concat(
        [pd.read_csv(f) for f in metric_files], ignore_index=True
    )
    metrics_combined.to_csv(os.path.join(output_dir, '_all_metrics.csv'), index=False)
    print(f'Metrics:      {len(metric_files)} files → {len(metrics_combined)} rows')

# Assemble daily counts
daily_files = sorted(glob.glob(os.path.join(daily_dir, '[!_]*_daily.csv')))
if daily_files:
    daily_combined = pd.concat(
        [pd.read_csv(f) for f in daily_files], ignore_index=True
    )
    daily_combined.to_csv(os.path.join(output_dir, '_all_daily_counts.csv'), index=False)
    print(f'Daily counts: {len(daily_files)} files → {len(daily_combined)} rows')

In [ ]:
# ECOREGION-LEVEL QUALITY METRICS ------------------------------------------------------------------
# Computed across years per ecoregion from the assembled files.
# Items: CV of peak DOY, interannual profile correlation, ecoregion bimodal flag.
#
# NOTE: pct_years_valid (already in _all_metrics.csv) covers fraction of years above
#       detection threshold — no new work needed.
#
# Reads from: _all_metrics.csv and _all_daily_counts.csv
# Writes to:  _eco_quality.csv

metrics_df = pd.read_csv(os.path.join(output_dir, '_all_metrics.csv'))
daily_df   = pd.read_csv(os.path.join(output_dir, '_all_daily_counts.csv'))

print(f'Loaded {len(metrics_df)} ecoregion-year rows across '
      f'{metrics_df["eco_id"].nunique()} ecoregions.')

# -----------------------------------------------------------------------
# CV of peak DOY across years
# Standard deviation / mean of peak_doy across all valid years.
# High CV = peak date is erratic year-to-year.
# -----------------------------------------------------------------------
cv_df = (
    metrics_df.groupby('eco_id')['peak_doy']
    .agg(cv_peak_doy=lambda x: round(float(x.std() / x.mean()), 4)
                                if len(x) > 1 and x.mean() != 0 else None)
    .reset_index()
)

# -----------------------------------------------------------------------
# Ecoregion-level bimodal flag
# Aggregates per-year bimodal_flag_year across all valid years.
# Flag = 1 if > 30% of years were flagged, 0 otherwise.
# -----------------------------------------------------------------------
def eco_bimodal_flag(flags):
    frac_flagged = (flags == 1).sum() / len(flags)
    return 1 if frac_flagged > 0.30 else 0

flag_df = (
    metrics_df.groupby('eco_id')['bimodal_flag_year']
    .agg(
        frac_flagged     = lambda x: round(float((x == 1).sum() / len(x)), 3),
        bimodal_flag_eco = eco_bimodal_flag
    )
    .reset_index()
)

flag_summary = flag_df['bimodal_flag_eco'].value_counts().sort_index()
print(f'\nEcoregion bimodal flag summary:')
print(f'  Clean   (0) : {flag_summary.get(0, 0)}')
print(f'  Flagged (1) : {flag_summary.get(1, 0)}')

# -----------------------------------------------------------------------
# Interannual profile correlation
# For each ecoregion: correlate each year's daily detection profile
# against the long-term mean profile, then average those correlations.
# High mean correlation = consistent season shape year-to-year = reliable metrics.
# Requires >= 3 valid years to compute meaningfully.
# -----------------------------------------------------------------------
profile_corr_rows = []

for eco_id, eco_daily in daily_df.groupby('eco_id'):

    pivot = eco_daily.pivot_table(
        index='year', columns='doy',
        values='n_detections', fill_value=0
    )

    if len(pivot) < 3:
        profile_corr_rows.append({'eco_id': eco_id, 'mean_profile_corr': None})
        continue

    mean_profile = pivot.mean(axis=0).values

    corrs = []
    for yr in pivot.index:
        yr_profile = pivot.loc[yr].values
        if yr_profile.sum() > 0 and mean_profile.sum() > 0:
            r = float(np.corrcoef(yr_profile, mean_profile)[0, 1])
            if not np.isnan(r):
                corrs.append(r)

    mean_corr = round(float(np.mean(corrs)), 4) if corrs else None
    profile_corr_rows.append({'eco_id': eco_id, 'mean_profile_corr': mean_corr})

corr_df = pd.DataFrame(profile_corr_rows)

# -----------------------------------------------------------------------
# MERGE AND SAVE
# -----------------------------------------------------------------------
eco_quality = (
    cv_df
    .merge(flag_df,  on='eco_id')
    .merge(corr_df,  on='eco_id')
)

eco_quality_path = os.path.join(output_dir, '_eco_quality.csv')
eco_quality.to_csv(eco_quality_path, index=False)

print(f'\nEcoregion quality metrics saved: {len(eco_quality)} ecoregions')
print(f'Path: {os.path.abspath(eco_quality_path)}')
print()
print(eco_quality.head(10).to_string())

In [ ]:
# COMBINE ALL RESULTS INTO MASTER CSV --------------------------------------------------------------
# Reads from _all_metrics.csv and _eco_quality.csv (both on disk).
# Ecoregion-level quality columns are broadcast to every row for that ecoregion.

master_df   = pd.read_csv(os.path.join(output_dir, '_all_metrics.csv'))
eco_quality = pd.read_csv(os.path.join(output_dir, '_eco_quality.csv'))

master_df = master_df.merge(eco_quality, on='eco_id', how='left')

col_order = [
    'eco_id', 'eco_name', 'biome_num', 'biome_name', 'year',
    # Primary metrics
    'onset_doy', 'peak_doy', 'end_doy', 'season_length',
    'n_detections', 'onset_month', 'peak_month',
    # Alternative thresholds
    'onset_doy_10', 'end_doy_90',
    'onset_doy_15', 'end_doy_85',
    # Profile shape
    'median_doy', 'mean_median_div', 'q25_doy', 'q75_doy',
    'iqr_season_length', 'active_days',
    'peak_concentration', 'skewness', 'kurtosis_raw',
    # Bimodality diagnostics (per year)
    'bc', 'bimodal_flag_year',
    # Per-year quality
    'peak_outside_window', 'n_years_valid', 'pct_years_valid',
    # Ecoregion-level quality
    'cv_peak_doy', 'mean_profile_corr',
    'frac_flagged', 'bimodal_flag_eco',
]

col_order = [c for c in col_order if c in master_df.columns]
master_df = master_df[col_order]

master_path = os.path.join(output_dir, f'master_{_run_name}.csv')
master_df.to_csv(master_path, index=False)

print(f'Master CSV: {master_df.shape[0]} rows × {master_df.shape[1]} columns')
print(f'Path: {os.path.abspath(master_path)}')
print()
print(master_df.head(10).to_string())